# Integrity checks

The routines in this notebook perform basic checks on the selected datasets that are easier to be made programmatic:
- checking for NaNs
- checking for aphysical quantities
- dimensions of datasets are what we expect them to be

Eventually it could be useful to operationalize these tests into the production system to make it automated. For now we'll have a section that performs integrity checks on the input datasets and one evaluates output datasets.

In [1]:
%load_ext autoreload
%autoreload 2

from srm import catalog
from srm.qaqc import check_physical_constraints, confirm_coords

In [2]:
catalog

Dataset Catalog (10 datasets)
+---------------------------------+----------+-----------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------+
| Name                            | Format   | Path                                                                                                            | Expected Chunks                                             |
+=================================+==========+=================================================================================================================+=============================================================+
| CESM2-WACCM-Historical-icechunk | icechunk | s3://carbonplan-srm/input/tensor/CESM2/CESM2-WACCM-Historical/icechunk/CESM2-WACCM-Historical_pancakes.icechunk | {'time': 30, 'lat': 192, 'lon': 288}                        |
+---------------------------------+----------+--------------------------------

# Get input data

Once the input datasets are in their final place for use and are all done with any pre-processing (e.g. rechunking), they won't change. So, we only have to run the integrity routines once and they can live in their own separate notebook.

### Output from three climate model simulations

These simulations were conducted in the Community Earth System Model version 2 -- Whole Atmosphere Community Climate Model (CESM2-WACCM)
* Historical (1978 - 2014)
* SSP245 (2015 - 2070)
* G6-1.5K (2035 - 2085)

In [16]:
gcm = "CESM2-WACCM"
scenario = "SSP245"  # "G6-1.5K" , "Historical"
variables = ["tas", "tasmin", "tasmax", "pr", "rsds"]
dataset_name = f"{gcm}-{scenario}-icechunk"
dataset_name

'CESM2-WACCM-SSP245-icechunk'

In [14]:
ds = catalog.get(dataset_name).to_xarray()
ds

<xarray.Dataset> Size: 625GB
Dimensions:          (ensemble_member: 9, time: 31391, lat: 192, lon: 288,
                      bounds: 2)
Coordinates:
  * ensemble_member  (ensemble_member) object 72B '001' '002' ... '009' '010'
  * time             (time) datetime64[ns] 251kB 2015-01-01 ... 2101-01-01
  * lat              (lat) float64 2kB -90.0 -89.06 -88.12 ... 88.12 89.06 90.0
  * lon              (lon) float64 2kB -180.0 -178.8 -177.5 ... 177.5 178.8
    lat_bounds       (lat, bounds) float64 3kB dask.array<chunksize=(192, 2), meta=np.ndarray>
    lon_bounds       (lon, bounds) float64 5kB dask.array<chunksize=(288, 2), meta=np.ndarray>
    time_bounds      (time, bounds) datetime64[ns] 502kB dask.array<chunksize=(480, 2), meta=np.ndarray>
Dimensions without coordinates: bounds
Data variables:
    huss             (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
    rlds             (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
    ps               (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
    hurs             (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
    pr               (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
    tas              (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
    tasmax           (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
    tasmin           (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
    sfcWind          (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
    rsds             (ensemble_member, time, lat, lon) float32 62GB dask.array<chunksize=(1, 30, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.8
    source:            CAM
    case:              b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001
    logname:           cmip6
    host:              cheyenne4
    initial_file:      b.e21.BWHIST.f09_g17.CMIP6-historical-WACCM.001.cam.i....
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1
    scenario:          SSP245
    model:             CESM2-WACCM

### Observations: ERA5

In [15]:
era5 = catalog.get("ERA5").to_xarray()
era5

<xarray.Dataset> Size: 789GB
Dimensions:      (time: 23741, lat: 721, lon: 1440, bounds: 2)
Coordinates:
  * time         (time) datetime64[ns] 190kB 1950-01-01 ... 2014-12-31
  * lat          (lat) float32 3kB -90.0 -89.75 -89.5 -89.25 ... 89.5 89.75 90.0
  * lon          (lon) float32 6kB -180.0 -179.8 -179.5 ... 179.2 179.5 179.8
    lat_bounds   (lat, bounds) float32 6kB dask.array<chunksize=(721, 2), meta=np.ndarray>
    lon_bounds   (lon, bounds) float32 12kB dask.array<chunksize=(1440, 2), meta=np.ndarray>
    time_bounds  (time, bounds) datetime64[ns] 380kB dask.array<chunksize=(30, 2), meta=np.ndarray>
Dimensions without coordinates: bounds
Data variables:
    ps           (time, lat, lon) float32 99GB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    rlds         (time, lat, lon) float32 99GB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    pr           (time, lat, lon) float32 99GB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    rsds         (time, lat, lon) float32 99GB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tas          (time, lat, lon) float32 99GB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tasmin       (time, lat, lon) float32 99GB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    sfcWind      (time, lat, lon) float32 99GB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    tasmax       (time, lat, lon) float32 99GB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    last_updated:           2026-01-28 02:05:04.838951+00:00
    valid_time_start:       1950-01-01
    valid_time_stop:        2014-12-31
    valid_time_stop_era5t:  2026-01-22

### Integrity checks

In [17]:
# test running on a 2x2 box for expediency since the dataset is chunked along full time dimension
# and so would take a long time to check fully

# use ensemble_member = 005 until https://github.com/carbonplan/srm-downscaling/issues/72#issuecomment-3874531853 is resolved
out = ds.sel(lat=slice(46, 48), lon=slice(-124, -122.0)).sel(ensemble_member="005").load()

In [18]:
check_physical_constraints(out)

Number of negative precipitation values: 0
Number of outlandishly high precipitation values: 0
Number of outlandishly high tas values: 0
Number of outlandishly high tasmax values: 0
Number of outlandishly high tasmin values: 0
Number of outlandishly low tas values: 0
Number of outlandishly low tasmax values: 0
Number of outlandishly low tasmin values: 0
Number of times tasmin exceeds tas: 0
Number of times tas exceeds tasmax: 0
Number of times tasmin exceeds tasmax: 0


In [20]:
check_physical_constraints(era5.sel(lat=slice(47, 48), lon=slice(-122, -121)))

Number of negative precipitation values: 0
Number of outlandishly high precipitation values: 0
Number of outlandishly high tas values: 0
Number of outlandishly high tasmax values: 0
Number of outlandishly high tasmin values: 0
Number of outlandishly low tas values: 0
Number of outlandishly low tasmax values: 0
Number of outlandishly low tasmin values: 0
Number of times tasmin exceeds tas: 89
Number of times tas exceeds tasmax: 5440
Number of times tasmin exceeds tasmax: 0


In [ ]:
confirm_coords(era5)